# 04 - Exports and Interactive Map

In [1]:
import sys
sys.path.insert(0, '..')

import json
import shutil

import folium
import geopandas as gpd
import pandas as pd
from shapely import set_precision

from src.crashes import aggregate_crashes_to_segments, attach_crash_counts, load_crash_data
from src.features import compute_mapillary_url
from src.road_names import backfill_road_names, load_overture_names
from src.score import INSUFFICIENT_DATA_TIER
from src.utils import METRIC_CRS

reliable = gpd.read_parquet('../data/processed/_reliable_scored.parquet')
reliable = gpd.GeoDataFrame(reliable, geometry='geometry', crs='EPSG:4326')
low_confidence = gpd.read_parquet('../data/processed/_low_confidence_final.parquet')
low_confidence = gpd.GeoDataFrame(low_confidence, geometry='geometry', crs='EPSG:4326')
with open('../data/processed/_sensitivity_summary.json') as f:
    sens_summary = json.load(f)

# road_name: names_primary (India) / english_ro (Thailand), falling back to a
# class + id label where neither source field is populated. Both raw columns
# already exist on these frames (harmonize_schema copies the source geojson
# through unchanged), so this can be derived without re-running notebooks 01-03.
def add_road_name(gdf):
    name = gdf['names_primary'] if 'names_primary' in gdf.columns else pd.Series(pd.NA, index=gdf.index)
    if 'english_ro' in gdf.columns:
        name = name.where(name.notna() & (name.astype(str).str.strip() != ''), gdf['english_ro'])
    has_name = name.notna() & (name.astype(str).str.strip() != '')
    fallback = gdf['road_class'].fillna('Road') + ' segment ' + gdf['segment_id']
    gdf['road_name'] = name.where(has_name, fallback)
    gdf['road_name_is_fallback'] = ~has_name
    return gdf

reliable = add_road_name(reliable)
low_confidence = add_road_name(low_confidence)

name_before = {
    country: {
        'reliable_pct': round((~reliable.loc[reliable['country'] == country, 'road_name_is_fallback']).mean() * 100, 1),
        'low_confidence_pct': round((~low_confidence.loc[low_confidence['country'] == country, 'road_name_is_fallback']).mean() * 100, 1),
    }
    for country in ['India', 'Thailand']
}

# Backfill segments the raw exports left unnamed using Overture Maps' own road
# names (fetched via scratchpad/fetch_overture_names.py, saved to data/raw/
# since it needs a live download -- see src/road_names.py for why this is a
# modest, not complete, improvement).
overture_paths = {'India': '../data/raw/overture_road_names_india.parquet',
                   'Thailand': '../data/raw/overture_road_names_thailand.parquet'}
for country, path in overture_paths.items():
    overture_names = load_overture_names(path)
    for gdf_name in ['reliable', 'low_confidence']:
        gdf = reliable if gdf_name == 'reliable' else low_confidence
        mask = gdf['country'] == country
        backfilled = backfill_road_names(gdf.loc[mask], overture_names, METRIC_CRS[country])
        gdf.loc[mask, ['road_name', 'road_name_is_fallback']] = backfilled[['road_name', 'road_name_is_fallback']]

name_after = {
    country: {
        'reliable_pct': round((~reliable.loc[reliable['country'] == country, 'road_name_is_fallback']).mean() * 100, 1),
        'low_confidence_pct': round((~low_confidence.loc[low_confidence['country'] == country, 'road_name_is_fallback']).mean() * 100, 1),
    }
    for country in ['India', 'Thailand']
}
print('Named-segment coverage before -> after Overture backfill:')
for country in ['India', 'Thailand']:
    print(f"  {country} reliable:        {name_before[country]['reliable_pct']}% -> {name_after[country]['reliable_pct']}%")
    print(f"  {country} low-confidence:  {name_before[country]['low_confidence_pct']}% -> {name_after[country]['low_confidence_pct']}%")

Named-segment coverage before -> after Overture backfill:
  India reliable:        29.2% -> 32.6%
  India low-confidence:  28.3% -> 38.4%
  Thailand reliable:        30.6% -> 48.9%
  Thailand low-confidence:  7.0% -> 61.5%


## Export `segments_scored.geojson` / `.gpkg` and `summary_statistics.csv`

In [2]:
reliable.to_file('../outputs/segments_scored.geojson', driver='GeoJSON')
reliable.to_file('../outputs/segments_scored.gpkg', driver='GPKG', layer='segments_scored')
print(f'Saved segments_scored.geojson/.gpkg ({len(reliable)} segments)')

total = len(reliable) + len(low_confidence)
tier_counts = reliable['risk_tier'].value_counts()
tier_counts[INSUFFICIENT_DATA_TIER] = len(low_confidence)

rows = [
    {'metric': 'total_segments', 'value': total},
    {'metric': 'reliable_segments', 'value': len(reliable)},
    {'metric': 'low_confidence_segments', 'value': len(low_confidence)},
]
for tier, count in tier_counts.items():
    key = tier.replace(' ', '_')
    rows.append({'metric': f'risk_tier_count_{key}', 'value': count})
    rows.append({'metric': f'risk_tier_pct_{key}', 'value': round(count / total * 100, 2)})
rows += [
    {'metric': 'mean_speed_safety_score', 'value': round(reliable['speed_safety_score'].mean(), 2)},
    {'metric': 'max_speed_safety_score', 'value': reliable['speed_safety_score'].max()},
    {'metric': 'mean_speed_gap_kmh', 'value': round(reliable['speed_gap'].mean(), 2)},
    {'metric': 'correlation_speed_safety_score_vs_ranked_percentile', 'value': round(sens_summary['correlation_with_ranked_percentile'], 4)},
    {'metric': 'sensitivity_avg_top20pct_overlap_pct', 'value': round(sens_summary['average_overlap_pct'], 2)},
    {'metric': 'mean_speed_limit_gap_kmh', 'value': round(reliable['speed_limit_gap'].mean(), 2)},
    {'metric': 'pct_segments_posted_above_recommendation', 'value': round((reliable['speed_limit_gap'] > 0).mean() * 100, 1)},
]
summary_df = pd.DataFrame(rows)
summary_df.to_csv('../outputs/summary_statistics.csv', index=False)
display(summary_df)

Saved segments_scored.geojson/.gpkg (14546 segments)


,metric,value
0,total_segments,69966.0000
1,reliable_segments,14546.0000
2,low_confidence_segments,55420.0000
3,risk_tier_count_Low_risk,8991.0000
4,risk_tier_pct_Low_risk,12.8500
5,risk_tier_count_Medium_risk,5476.0000
6,risk_tier_pct_Medium_risk,7.8300
7,risk_tier_count_High_risk,79.0000
8,risk_tier_pct_High_risk,0.1100
9,risk_tier_count_Insufficient_data,55420.0000


## Validate the Speed Safety Score against real 2025 Thailand crash data

`data/raw/Thailandaccident2025.xlsx` is the Department of Highways' 2025
accident log: 23,715 geocoded crash records with fatality/injury counts. This
is the first segment-joinable crash outcome data available to this project
-- the ATO Road Safety workbook (`src/train_model.py`) is national-level
only, one row per country per year, so it can't validate individual segment
scores.

**This data is used for validation and map enrichment, not folded into the
weighted score.** The score's absolute anchors are deliberately built so a
score means the same thing in India and Thailand; crash records exist for
Thailand only, so adding them as a scoring term would make Thailand's
formula structurally different from India's and break that guarantee. See
`src/crashes.py` for the join logic (nearest-match within 300m).

In [3]:
CRASH_COLUMNS = ['crash_count', 'fatality_count', 'serious_injury_count', 'minor_injury_count', 'ksi_count']

crashes = load_crash_data('../data/raw/Thailandaccident2025.xlsx')
th_reliable = reliable[reliable['country'] == 'Thailand']
crash_agg = aggregate_crashes_to_segments(crashes, th_reliable)

reliable = attach_crash_counts(reliable, crash_agg)
# Not joined to low_confidence: crash_agg only matched against reliable Thailand
# segments, so every low-confidence segment_id would zero-fill as "no crashes"
# rather than "not checked" -- misleadingly implying data that isn't there.

# Re-save the exports now that crash columns are attached.
reliable.to_file('../outputs/segments_scored.geojson', driver='GeoJSON')
reliable.to_file('../outputs/segments_scored.gpkg', driver='GPKG', layer='segments_scored')

th = reliable[reliable['country'] == 'Thailand'].copy()
th['crash_rate_per_km'] = th['crash_count'] / th['RoadLength_km'].clip(lower=0.1)
th['ksi_rate_per_km'] = th['ksi_count'] / th['RoadLength_km'].clip(lower=0.1)
th['fatality_rate_per_km'] = th['fatality_count'] / th['RoadLength_km'].clip(lower=0.1)
has_crash = th['crash_count'] > 0
th['ksi_severity'] = th['ksi_count'].where(has_crash) / th['crash_count'].where(has_crash)

tier_validation = th.groupby('risk_tier')[['crash_rate_per_km', 'ksi_rate_per_km', 'fatality_rate_per_km']].mean()
tier_severity = th.loc[has_crash].groupby('risk_tier')['ksi_severity'].mean()
tier_validation['ksi_severity_given_crash'] = tier_severity

crash_validation_summary = {
    'n_crashes_total': int(crashes.attrs['n_total']),
    'n_crashes_dropped_invalid_coords': int(crashes.attrs['n_dropped_invalid_coords']),
    'n_crashes_matched': int(crash_agg.attrs['n_crashes_matched']),
    'match_rate_pct': crash_agg.attrs['match_rate_pct'],
    'n_thailand_segments_with_a_crash': int(has_crash.sum()),
    'pct_thailand_segments_with_a_crash': round(has_crash.mean() * 100, 1),
    'corr_score_vs_crash_rate_per_km': round(th['speed_safety_score'].corr(th['crash_rate_per_km']), 4),
    'corr_score_vs_ksi_rate_per_km': round(th['speed_safety_score'].corr(th['ksi_rate_per_km']), 4),
    'corr_score_vs_fatality_rate_per_km': round(th['speed_safety_score'].corr(th['fatality_rate_per_km']), 4),
    'tier_means': tier_validation.round(4).to_dict(orient='index'),
}
with open('../data/processed/_crash_validation_summary.json', 'w') as f:
    json.dump(crash_validation_summary, f, indent=2)

crash_rows = [
    {'metric': 'crashes_2025_total', 'value': crash_validation_summary['n_crashes_total']},
    {'metric': 'crashes_2025_matched_to_segment', 'value': crash_validation_summary['n_crashes_matched']},
    {'metric': 'crash_match_rate_pct', 'value': crash_validation_summary['match_rate_pct']},
    {'metric': 'pct_thailand_segments_with_a_2025_crash', 'value': crash_validation_summary['pct_thailand_segments_with_a_crash']},
    {'metric': 'corr_score_vs_crash_rate_per_km', 'value': crash_validation_summary['corr_score_vs_crash_rate_per_km']},
    {'metric': 'corr_score_vs_ksi_rate_per_km', 'value': crash_validation_summary['corr_score_vs_ksi_rate_per_km']},
    {'metric': 'corr_score_vs_fatality_rate_per_km', 'value': crash_validation_summary['corr_score_vs_fatality_rate_per_km']},
]
summary_df = pd.concat([summary_df, pd.DataFrame(crash_rows)], ignore_index=True)
summary_df.to_csv('../outputs/summary_statistics.csv', index=False)

print(f"Matched {crash_validation_summary['n_crashes_matched']} / {crash_validation_summary['n_crashes_total']} crashes "
      f"to a Thailand segment ({crash_validation_summary['match_rate_pct']}%)")
print(f"{crash_validation_summary['n_thailand_segments_with_a_crash']} Thailand segments "
      f"({crash_validation_summary['pct_thailand_segments_with_a_crash']}%) have at least one recorded 2025 crash")
print('\nRaw score correlations (weak -- crash frequency is driven by traffic volume, not risk):')
print(f"  score vs crash_rate_per_km:    {crash_validation_summary['corr_score_vs_crash_rate_per_km']}")
print(f"  score vs ksi_rate_per_km:      {crash_validation_summary['corr_score_vs_ksi_rate_per_km']}")
print(f"  score vs fatality_rate_per_km: {crash_validation_summary['corr_score_vs_fatality_rate_per_km']}")
print('\nMean outcome rates by risk tier (the score predicts severity, not frequency):')
display(tier_validation)

Matched 20758 / 23715 crashes to a Thailand segment (88.2%)
4653 Thailand segments (40.4%) have at least one recorded 2025 crash

Raw score correlations (weak -- crash frequency is driven by traffic volume, not risk):
  score vs crash_rate_per_km:    -0.0211
  score vs ksi_rate_per_km:      0.0048
  score vs fatality_rate_per_km: 0.003

Mean outcome rates by risk tier (the score predicts severity, not frequency):


,crash_rate_per_km,ksi_rate_per_km,fatality_rate_per_km,ksi_severity_given_crash
risk_tier,,,,
High risk,0.423060,0.239675,0.187183,0.558816
Low risk,0.439013,0.089743,0.042839,0.291909
Medium risk,0.402125,0.101730,0.047333,0.296778


## Build the interactive Folium map

Reliable (scored) segments get full tooltips and a clickable Mapillary popup. Low-confidence segments are shown as a separate, lighter-weight, off-by-default layer (coarser geometry simplification, minimal tooltip, no popup) -- embedding all ~70k segments at full fidelity produced an unusably large (60MB+) HTML file, so the low-confidence layer trades some visual fidelity for a browser-friendly file size.

In [4]:
RISK_COLORS = {
    'High risk': '#E24B4A', 'Medium risk': '#EF9F27',
    'Low risk': '#1D9E75', INSUFFICIENT_DATA_TIER: '#B4B2A9',
}
TIER_CLASSES = {
    'High risk': 'tier-hi', 'Medium risk': 'tier-med',
    'Low risk': 'tier-lo', INSUFFICIENT_DATA_TIER: 'tier-na',
}
# Hover/select colors are deliberately outside the risk-tier palette above so a
# highlighted segment is unambiguous regardless of which tier it belongs to.
HOVER_COLOR = '#00B8D9'   # cyan: transient, while the pointer is over a segment
SELECT_COLOR = '#7C3AED'  # violet: persists after a click, until another segment is clicked

LOW_CONF_TOOLTIP_FIELDS = ['road_name', 'road_class', 'risk_tier', 'RoadLength_km', 'Sample_Size_Total']
LOW_CONF_TOOLTIP_ALIASES = ['Road:', 'Class:', 'Risk tier:', 'Road length (km):', 'Sample size:']

# One shared stylesheet (injected once into the map, see the map-building cell)
# instead of repeating inline CSS on every one of 14k+ tooltip/popup strings --
# that repetition is what pushed a first draft of this map to 100MB+ and over
# GitHub's 100MB per-file push limit.
SEGMENT_CARD_CSS = '''
<style>
  .seg-tt, .seg-pop { font-family: -apple-system, "Segoe UI", Roboto, sans-serif; }
  .seg-tt { min-width: 160px; }
  .seg-name { font-size: 13.5px; font-weight: 700; margin-bottom: 3px; }
  .seg-pop .seg-name { font-size: 15px; margin-bottom: 2px; }
  .seg-sub { font-size: 11px; color: #888; margin-bottom: 8px; }
  .badge { display: inline-block; padding: 2px 8px; border-radius: 10px; color: #fff; font-size: 11px; font-weight: 600; }
  .seg-pop .badge { padding: 3px 10px; border-radius: 12px; font-size: 12px; margin-bottom: 8px; }
  .tier-hi { background: #E24B4A; } .tier-med { background: #EF9F27; }
  .tier-lo { background: #1D9E75; } .tier-na { background: #B4B2A9; }
  .seg-pop { min-width: 230px; max-width: 280px; }
  .seg-table { width: 100%; font-size: 12.5px; border-collapse: collapse; margin-top: 8px; }
  .seg-table td { padding: 2px 10px 2px 0; white-space: nowrap; }
  .seg-table td:first-child { color: #666; }
  .seg-table td:last-child { text-align: right; font-weight: 600; }
  .gap-pos { color: #E24B4A; } .gap-neg { color: #1D9E75; }
  .seg-crash { margin-top: 9px; padding-top: 8px; border-top: 1px solid #e5e5e5; font-size: 12px; }
  .seg-link { margin-top: 9px; font-size: 12px; }
</style>
'''


def _tooltip_html(r):
    """Compact hover card: just enough to identify the segment and its headline risk."""
    return (f'<div class="seg-tt"><div class="seg-name">{r["road_name"]}</div>'
            f'<span class="badge {TIER_CLASSES.get(r["risk_tier"], "tier-na")}">{r["risk_tier"]} &middot; {r["speed_safety_score"]:.0f}</span></div>')


def _popup_html(r):
    """Full detail card shown on click: road identity, risk badge, and all key metrics in a readable layout."""
    gap_class = 'gap-pos' if r['speed_limit_gap'] > 0 else 'gap-neg'
    rows = (
        f'<tr><td>Speed limit</td><td>{r["SpeedLimit"]:.0f} km/h</td></tr>'
        f'<tr><td>85th pct. speed</td><td>{r["F85thPercentileSpeed"]:.0f} km/h</td></tr>'
        f'<tr><td>Speed gap</td><td>{"+" + format(r["speed_gap"], ".0f") + " km/h" if r["speed_gap"] > 0 else "0 km/h"}</td></tr>'
        f'<tr><td>Recommended limit</td><td>{format(r["recommended_speed_limit"], ".0f") + " km/h" if pd.notna(r["recommended_speed_limit"]) else "n/a"}</td></tr>'
        f'<tr><td>Limit vs. recommended</td><td class="{gap_class}">{format(r["speed_limit_gap"], "+.0f") + " km/h" if pd.notna(r["speed_limit_gap"]) else "n/a"}</td></tr>'
        f'<tr><td>Road length</td><td>{r["RoadLength_km"]:.1f} km</td></tr>'
        f'<tr><td>Sample size</td><td>{r["Sample_Size_Total"]:.0f}</td></tr>'
    )

    crash_block = ''
    if r['country'] == 'Thailand' and pd.notna(r.get('crash_count')):
        if r['crash_count'] > 0:
            crash_block = (f'<div class="seg-crash"><b>{r["crash_count"]:.0f}</b> recorded crashes in 2025 within 300m &middot; '
                            f'<b class="gap-pos">{r["fatality_count"]:.0f} fatal</b>, {r["ksi_count"]:.0f} killed/seriously injured</div>')
        else:
            crash_block = '<div class="seg-crash">No crashes recorded nearby in 2025</div>'

    street_view = ''
    if pd.notna(r.get('mapillary_url')):
        street_view = f'<div class="seg-link"><a href="{r["mapillary_url"]}" target="_blank" rel="noopener">View street imagery &rarr;</a></div>'

    return (
        f'<div class="seg-pop"><div class="seg-name">{r["road_name"]}</div>'
        f'<div class="seg-sub">{r["road_class"]} &middot; {r["country"]} &middot; segment {r["segment_id"]}</div>'
        f'<span class="badge {TIER_CLASSES.get(r["risk_tier"], "tier-na")}">{r["risk_tier"]} &middot; score {r["speed_safety_score"]:.0f}</span>'
        f'<table class="seg-table">{rows}</table>{crash_block}{street_view}</div>'
    )

In [5]:
RELIABLE_COLS = ['segment_id', 'road_name', 'road_class', 'SpeedLimit', 'F85thPercentileSpeed',
                 'speed_gap', 'recommended_speed_limit', 'speed_limit_gap', 'risk_tier',
                 'speed_safety_score', 'RoadLength_km', 'Sample_Size_Total', 'country',
                 'StreetImageLink'] + CRASH_COLUMNS
LOW_CONF_COLS = LOW_CONF_TOOLTIP_FIELDS + ['country']


def _prep_reliable(reliable):
    cols = RELIABLE_COLS + ['geometry']
    out = reliable[cols].copy()
    out['geometry'] = out.geometry.simplify(0.0003, preserve_topology=True)
    out['geometry'] = gpd.GeoSeries(set_precision(out.geometry.values, grid_size=0.00001), crs='EPSG:4326')
    for c in ['SpeedLimit', 'F85thPercentileSpeed', 'speed_gap', 'RoadLength_km']:
        out[c] = pd.to_numeric(out[c], errors='coerce').round(1)
    out['speed_safety_score'] = pd.to_numeric(out['speed_safety_score'], errors='coerce').round(1)
    out['Sample_Size_Total'] = pd.to_numeric(out['Sample_Size_Total'], errors='coerce').round(0)
    out = compute_mapillary_url(out)
    out['tooltip'] = out.apply(_tooltip_html, axis=1)
    out['popup'] = out.apply(_popup_html, axis=1)
    return out.drop(columns=['mapillary_url', 'StreetImageLink'] + CRASH_COLUMNS)

def _prep_low_confidence(low_confidence):
    # Kept lightweight on purpose: this layer covers ~55k geometrically coarser
    # segments and is off by default, so it uses Folium's plain fields/aliases
    # tooltip (values only, one shared render template) rather than a custom
    # HTML card per feature -- see "Map performance trade-off" in methodology.md.
    low_confidence = low_confidence.copy()
    low_confidence['risk_tier'] = INSUFFICIENT_DATA_TIER
    cols = LOW_CONF_COLS + ['geometry']
    out = low_confidence[cols].copy()
    out['geometry'] = out.geometry.simplify(0.0015, preserve_topology=True)
    out['geometry'] = gpd.GeoSeries(set_precision(out.geometry.values, grid_size=0.0001), crs='EPSG:4326')
    out['RoadLength_km'] = pd.to_numeric(out['RoadLength_km'], errors='coerce').round(1)
    out['Sample_Size_Total'] = pd.to_numeric(out['Sample_Size_Total'], errors='coerce').round(0)
    return out

reliable_map = _prep_reliable(reliable)
low_conf_map = _prep_low_confidence(low_confidence)

In [6]:
all_bounds = pd.concat([reliable_map.geometry, low_conf_map.geometry]).total_bounds
center = [(all_bounds[1] + all_bounds[3]) / 2, (all_bounds[0] + all_bounds[2]) / 2]
m = folium.Map(location=center, zoom_start=5, tiles='CartoDB Positron')

title_html = '''
<div style="position: fixed; top: 12px; right: 12px; z-index: 9999; background: white;
            padding: 8px 14px; border-radius: 4px; box-shadow: 0 1px 4px rgba(0,0,0,0.4);
            font-size: 16px; font-weight: 600;">
  AI for Safer Roads &mdash; Speed Safety Score
</div>
'''
m.get_root().html.add_child(folium.Element(title_html))

legend_html = f'''
<div style="position: fixed; bottom: 24px; left: 24px; z-index: 9999; background: white;
            padding: 10px 14px; border-radius: 4px; box-shadow: 0 1px 4px rgba(0,0,0,0.4); font-size: 13px;">
  <b>Risk tier</b><br>
  <span style="display:inline-block;width:12px;height:12px;background:#E24B4A;margin-right:6px;"></span>High risk<br>
  <span style="display:inline-block;width:12px;height:12px;background:#EF9F27;margin-right:6px;"></span>Medium risk<br>
  <span style="display:inline-block;width:12px;height:12px;background:#1D9E75;margin-right:6px;"></span>Low risk<br>
  <span style="display:inline-block;width:12px;height:12px;background:#B4B2A9;margin-right:6px;"></span>Insufficient data<br>
  <hr style="margin:6px 0;border:none;border-top:1px solid #ddd;">
  <span style="display:inline-block;width:12px;height:12px;background:{SELECT_COLOR};margin-right:6px;"></span>Selected segment
</div>
'''
m.get_root().html.add_child(folium.Element(legend_html))

control_css = '''
<style>
  /* The Leaflet layer control defaults to top-right, same corner as the
     title bar above -- push it down so the title doesn't cover its top. */
  .leaflet-control-layers { margin-top: 70px !important; }
  .leaflet-popup-content-wrapper { border-radius: 8px; }
  .leaflet-popup-content { margin: 12px 14px; }
  path.leaflet-interactive { cursor: pointer; }
</style>
'''
m.get_root().html.add_child(folium.Element(control_css))
m.get_root().html.add_child(folium.Element(SEGMENT_CARD_CSS))

# Hover highlights a segment in HOVER_COLOR while the pointer is over it; a
# click locks the highlight in SELECT_COLOR until a different segment is
# clicked, independent of whichever risk-tier color it started as. Written as
# plain Leaflet event handlers (rather than folium's built-in highlight_function,
# which only supports transient hover) so both interactions share one selection
# state per map.
interactivity_js = f'''
<script>
(function() {{
  var HOVER_COLOR = "{HOVER_COLOR}";
  var SELECT_COLOR = "{SELECT_COLOR}";
  var selectedLayer = null;

  window.attachSegmentInteractivity = function(geojsonLayer) {{
    geojsonLayer.eachLayer(function(layer) {{
      layer._baseStyle = {{color: layer.options.color, weight: layer.options.weight, opacity: layer.options.opacity}};
      layer.on('mouseover', function(e) {{
        if (selectedLayer !== layer) {{
          layer.setStyle({{color: HOVER_COLOR, weight: 10, opacity: 1}});
          layer.bringToFront();
        }}
      }});
      layer.on('mouseout', function(e) {{
        if (selectedLayer !== layer) {{
          layer.setStyle(layer._baseStyle);
        }}
      }});
      layer.on('click', function(e) {{
        if (selectedLayer && selectedLayer !== layer) {{
          selectedLayer.setStyle(selectedLayer._baseStyle);
        }}
        layer.setStyle({{color: SELECT_COLOR, weight: 10, opacity: 1}});
        layer.bringToFront();
        selectedLayer = layer;
      }});
    }});
  }};
}})();
</script>
'''
m.get_root().html.add_child(folium.Element(interactivity_js))

def style_function(feature):
    tier = feature['properties']['risk_tier']
    return {'color': RISK_COLORS.get(tier, '#B4B2A9'), 'weight': 8 if tier == 'High risk' else 6, 'opacity': 0.85}

In [7]:
countries = sorted(set(reliable_map['country']) | set(low_conf_map['country']))
geojson_var_names = []

for country in countries:
    rel_sub = reliable_map[reliable_map['country'] == country]
    fg_rel = folium.FeatureGroup(name=f'{country} — scored segments', show=True)
    gj_rel = folium.GeoJson(
        rel_sub.__geo_interface__, style_function=style_function,
        tooltip=folium.GeoJsonTooltip(fields=['tooltip'], aliases=[''], labels=False, sticky=True, parse_html=True),
        popup=folium.GeoJsonPopup(fields=['popup'], aliases=[''], labels=False, parse_html=True, max_width=300),
    )
    gj_rel.add_to(fg_rel)
    fg_rel.add_to(m)
    geojson_var_names.append(gj_rel.get_name())

    lc_sub = low_conf_map[low_conf_map['country'] == country]
    fg_lc = folium.FeatureGroup(name=f'{country} — insufficient data', show=False)
    gj_lc = folium.GeoJson(
        lc_sub.__geo_interface__, style_function=style_function,
        tooltip=folium.GeoJsonTooltip(fields=LOW_CONF_TOOLTIP_FIELDS, aliases=LOW_CONF_TOOLTIP_ALIASES, sticky=True),
    )
    gj_lc.add_to(fg_lc)
    fg_lc.add_to(m)
    geojson_var_names.append(gj_lc.get_name())

folium.LayerControl(collapsed=False).add_to(m)

# Wire up hover/click highlighting once the page has fully loaded, so this
# doesn't depend on where Folium's per-layer script ends up relative to the
# attachSegmentInteractivity definition in the document.
attach_calls = '\n  '.join(
    f'if (typeof {name} !== "undefined") {{ window.attachSegmentInteractivity({name}); }}'
    for name in geojson_var_names
)
interactivity_call_js = f'''
<script>
window.addEventListener("load", function() {{
  {attach_calls}
}});
</script>
'''
m.get_root().html.add_child(folium.Element(interactivity_call_js))

m.save('../outputs/map.html')
shutil.copy('../outputs/map.html', '../docs/index.html')
print('Saved outputs/map.html and copied it to docs/index.html')
print('Countries on map:', countries)
print(f'Reliable (scored) features plotted: {len(reliable_map)}')
print(f'Low-confidence (insufficient data) features plotted: {len(low_conf_map)}')

Saved outputs/map.html and copied it to docs/index.html
Countries on map: ['India', 'Thailand']
Reliable (scored) features plotted: 14546
Low-confidence (insufficient data) features plotted: 55420


In [8]:
print('Map saved to outputs/map.html and docs/index.html — open in a browser to view (not rendered inline to keep this notebook small).')

Map saved to outputs/map.html and docs/index.html — open in a browser to view (not rendered inline to keep this notebook small).
